In [0]:
%sql
CREATE OR REPLACE TABLE medical_pipeline.gold.kpi_top_procedures AS
WITH proc_with_date AS (
  SELECT 
    p.*,
    d.year,
    d.month,
    d.quarter,
    d.half_year
  FROM medical_pipeline.gold.fact_procedures p
  LEFT JOIN medical_pipeline.gold.dim_dates d 
    ON p.procedure_date = d.date
),
agg_metrics AS (
  SELECT 
    year,
    half_year,
    procedure_code,
    AVG(base_cost) AS avg_cost,
    COUNT(*) AS procedure_count
  FROM proc_with_date
  GROUP BY year, half_year, procedure_code
),
ranked_procedures AS (
  SELECT 
    year,
    half_year,
    procedure_code,
    avg_cost,
    procedure_count,
    ROW_NUMBER() OVER (
      PARTITION BY year, half_year 
      ORDER BY avg_cost DESC
    ) AS rank
  FROM agg_metrics
)
SELECT 
procedure_code,
procedure_count,
  year,
  half_year,
  
  avg_cost,
  
  rank
FROM ranked_procedures
WHERE rank <= 10
ORDER BY year, half_year, rank;

In [0]:
%sql
select * from medical_pipeline.gold.kpi_top_procedures